# Step 6 — MVD Checkpoint

**The gate.** One line, all its SKUs, four economic outputs responding to the levers. If this works, the project is safe; everything after it is elaboration (architecture §5).

**Line under test: L3** (personal care, Plant 2, 15 SKUs). Chosen because it has zero overlap with the 14 SKUs Step 5a could not confidently bias-correct — building the MVD on L1 or L2 would conflate "is the engine working" with "is the unreliable bias input misbehaving." L3 also carries the highest margin (52%), so cost swings are visible.

**What the engine does.** Forward-simulates `horizon_months: 12` under a lever setting (D-038 — it projects the next year, it does not replay the last three). Demand is one deterministic path (D-041); lost sales come from the standard normal loss function applied to Step 5a's `irreducible_volatility_cv`, not from sampling futures.

**Logic lives in `src/engine.py`.** This notebook imports and calls it. Nothing is computed in a cell here.

**Prerequisites:** Step 5a's `demand_characteristics.csv` and `censoring_diagnostics.csv`.

## Setup — clone the repo

In [ ]:
import subprocess, os, sys

def sh(cmd, cwd=None):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    print(r.stdout[-3000:])
    if r.returncode != 0:
        print("STDERR:", r.stderr[-1500:])
    return r

REPO = '/content/ibp-tradeoff'
os.chdir('/content')
sh('rm -rf ibp-tradeoff')
sh('git clone https://github.com/rdelolmog-creator/ibp-tradeoff.git')
os.chdir(REPO)
sys.path.insert(0, REPO)
print('Working directory:', os.getcwd())

## Rebuild the Step 4 artefacts
Same approved pipeline, no code change. Produces `clean_master.parquet` **and** `sku_master.parquet` — the second is new at Step 6 (D-043): the engine needs price, standard cost, margin, shelf life, MOQ and min-run per SKU, and `app.py` cannot re-run the cleaner to get them.

Expect: `clean_master (2159, 13)` · `sku_master (60, 21)`

In [ ]:
import pandas as pd
from src.ingest import DataIngestor
from src.cleaner import DataCleaner

if not os.path.isdir('data_primary/raw'):
    sh('python generate_data.py')
    sh('mv data data_primary')

ing = DataIngestor(repo_root=REPO, data_root='data_primary')
raw = ing.load()
cl = DataCleaner(ing.schema, ing.assumptions)
clean_master, sku_master, _ = cl.clean(raw)

os.makedirs('data_primary/clean', exist_ok=True)
clean_master.to_parquet('data_primary/clean/clean_master.parquet', index=False)
sku_master.to_parquet('data_primary/clean/sku_master.parquet', index=False)

print('clean_master:', clean_master.shape, '| sku_master:', sku_master.shape)

## Upload Step 5a's output
Select **both** files from the blind chat's notebook run: `demand_characteristics.csv` and `censoring_diagnostics.csv`.

The second is needed for D-035: SKUs Step 5a marked `INCONCLUSIVE` receive zero bias correction whatever the lever says. That list is read programmatically from Step 5a's own verdicts, never re-derived here.

In [ ]:
from google.colab import files
print('Select demand_characteristics.csv AND censoring_diagnostics.csv:')
uploaded = files.upload()

demand_characteristics = pd.read_csv('demand_characteristics.csv').set_index('sku_id')
censoring_diagnostics = pd.read_csv('censoring_diagnostics.csv').set_index('sku_id')

from src.portfolio_impact import get_flagged_skus
flagged = get_flagged_skus(censoring_diagnostics.reset_index())

for f in ('demand_characteristics.csv', 'censoring_diagnostics.csv'):
    import shutil; shutil.copy(f, f'data_primary/clean/{f}')

print(f'demand_characteristics : {demand_characteristics.shape}')
print(f'flagged (INCONCLUSIVE) : {len(flagged)} SKUs')

## Build the engine

`build_line_master()` produces the `canonical.line_master` table that `schema.yaml` declares but no pipeline step emitted until now (D-044). Line economics are assumptions, not observed data — storing them as a generated artefact would create a second place to change them, so the table is built at runtime and validated against the schema declaration.

Expect: 15 SKUs on L3, and **zero** of them flagged.

In [ ]:
import yaml, numpy as np
from src.engine import TradeOffEngine, LeverSettings, build_line_master

assumptions = yaml.safe_load(open('config/assumptions.yaml'))
schema      = yaml.safe_load(open('config/schema.yaml'))

MVD_LINE = 'L3'

engine = TradeOffEngine(
    assumptions=assumptions,
    schema=schema,
    clean_master=clean_master,
    sku_master=sku_master,
    demand_characteristics=demand_characteristics,
    flagged_sku_ids=flagged,
)

print(engine.line_master.to_string(index=False))
print()
line_skus = engine.line_skus(MVD_LINE)
print(f'{MVD_LINE}: {len(line_skus)} SKUs  {line_skus}')
print(f'flagged on {MVD_LINE}: {sorted(set(line_skus) & set(flagged))}  '
      f'<- must be empty, that is why this line was chosen')
print(f'horizon: {engine.horizon_months} months   fingerprint: {engine.assumption_fingerprint}')

## The forward demand path

Derived from the 36 months of history alone — seasonal index by calendar month, OLS trend on the deseasonalised series, promo months excluded from level and trend.

**It reads nothing from `assumptions.yaml`.** The generator's own seasonality amplitude, phase and trend parameters live there; using them would hand the engine the answer the data is supposed to carry. This is the first of the three circularity defences in this notebook.

In [ ]:
import matplotlib.pyplot as plt

hist = (clean_master[clean_master.sku_id.isin(line_skus)]
        .groupby('month')['actual_units'].sum())
fwd  = engine.project_demand(MVD_LINE).groupby('month')['demand_units'].sum()

fig, ax = plt.subplots(figsize=(11, 3.6))
ax.plot(hist.index, hist.values, lw=1.4, label='history (36m, actual)')
ax.plot(fwd.index, fwd.values, lw=1.8, ls='--', label='forward path (12m, projected)')
ax.axvline(hist.index.max(), color='0.6', lw=0.8)
ax.set_title(f'{MVD_LINE} — total demand, history and forward horizon')
ax.set_ylabel('units'); ax.legend(); ax.grid(alpha=0.25)
plt.tight_layout(); plt.show()

print(fwd.round(0).to_string())

## Baseline scenario — all four outputs

Lever defaults come from `assumptions.yaml` (`abc.A.service_floor`, `abc.A.target_cover_weeks`, `categories.personal_care.min_run_hours`), never from literals in code.

**Read the excess/obsolescence line carefully.** It has two mechanisms: shelf-life write-off, and a slow-moving provision on stock beyond `inventory_risk.excess_cover_months_threshold`. On this line the write-off half is **structurally zero** — personal care's shelf life is 1,092 days and the write-off trigger is 10% of remaining life, so stock would have to sit ~32 months and the horizon is 12. That is D-008 working as designed, not a bug. The provision half is what makes the output live here; the write-off half is demonstrated live on L2 further down.

In [ ]:
category = str(sku_master.set_index('sku_id').loc[line_skus[0], 'category'])
base = LeverSettings.defaults(assumptions, category, 'A')
print('lever defaults:', base.as_dict())
print()

res = engine.run_scenario(MVD_LINE, base)

std_cost = engine.sku_master['std_cost_eur']
writeoff_eur = float((res.sku_month.writeoff_units * res.sku_month.sku_id.map(std_cost)).sum())

print(f'{"OUTPUT":<34}{"EUR / 12 months":>18}')
print('-' * 52)
for k in ('lost_sales_eur', 'excess_obsolescence_eur',
          'conversion_cost_eur', 'working_capital_cost_eur'):
    print(f'{k:<34}{res.totals[k]:>18,.0f}')
print('-' * 52)
print(f'{"total_economic_cost_eur":<34}{res.totals["total_economic_cost_eur"]:>18,.0f}')
print()
print(f'  of which shelf-life write-off  {writeoff_eur:>14,.0f}   <- structurally 0 on this line')
print(f'  of which slow-moving provision {res.totals["excess_obsolescence_eur"] - writeoff_eur:>14,.0f}')
print()
print(f'service achieved   {res.totals["service_achieved"]:.3f}')
print(f'mean utilisation   {res.totals["utilisation_mean"]:.3f}')
print(f'assumption set     {res.assumption_fingerprint}')

## The gate — one lever, four outputs

`min_run_hours` is the demonstration lever (D-040). It is expected to move three outputs in opposite directions at once — conversion cost down through fewer changeovers, carrying cost and slow-moving provision up through larger batches, lost sales down through the incidental buffer those batches create. That is a stronger demonstration than a single lever moving a single output, and it is the cross-lever coupling the model should reveal rather than hide.

**Nothing in the engine special-cases this. It has to emerge.**

In [ ]:
lo, hi = assumptions['levers']['min_run_hours']['range']
values = [2.0, 5.0, 9.0, 14.0, 20.0]

sweep = engine.sweep(MVD_LINE, 'min_run_hours', values, base)

show = ['lever_value', 'lost_sales_eur', 'excess_obsolescence_eur',
        'conversion_cost_eur', 'working_capital_cost_eur',
        'total_economic_cost_eur', 'service_achieved', 'utilisation_mean']
print(sweep[show].round(2).to_string(index=False))

best = sweep.loc[sweep.total_economic_cost_eur.idxmin()]
print(f'\nlowest total economic cost at min_run_hours = {best.lever_value:g} '
      f'({best.total_economic_cost_eur:,.0f} EUR)')

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(17, 3.2))
panels = [('lost_sales_eur', 'Lost sales'),
          ('excess_obsolescence_eur', 'Excess / slow-moving'),
          ('conversion_cost_eur', 'Conversion cost'),
          ('working_capital_cost_eur', 'Working capital'),
          ('total_economic_cost_eur', 'TOTAL')]
for ax, (col, title) in zip(axes, panels):
    ax.plot(sweep.lever_value, sweep[col] / 1e3, marker='o', lw=1.8)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('min_run_hours'); ax.grid(alpha=0.25)
axes[0].set_ylabel('EUR 000 / 12 months')
plt.tight_layout(); plt.show()

## The circularity guardrail

`assumptions.yaml`'s `guardrails` block is a statement of intent. This is the running test.

`forecast_bias_correction` moves alone, everything else held fixed. Every cost is then rebuilt from the simulated quantities at unit rates read before the sweep. If any cost can only be reproduced by referring to the lever value itself, the model is circular and every number it produces is an artefact of its own construction (§9).

Two structural facts make the test meaningful rather than decorative:
- the lever enters the engine in exactly one function, `project_plan()`, and it moves the **plan**, never demand;
- `demand_units` is asserted bit-identical across every lever setting.

In [ ]:
sh('python -m pytest tests/test_engine.py -v --no-header -x')

In [ ]:
carrying = assumptions['finance']['carrying_rate_monthly']
gm, sc = engine.sku_master['gross_margin_eur'], engine.sku_master['std_cost_eur']
share = engine.sku_master['category'].map(
    lambda k: assumptions['stockout_disposition'][k]['lost'])

print(f'{"lever":>7}{"working capital":>20}{"rebuilt":>16}{"lost sales":>16}{"rebuilt":>14}')
print('-' * 74)
for c in [0.0, 0.25, 0.5, 0.75, 1.0]:
    r = engine.run_scenario(MVD_LINE, base.replace(forecast_bias_correction=c))
    s = r.sku_month
    wc = float((s.stock_value_eur * carrying).sum())
    ls = float((s.lost_units * s.sku_id.map(share) * s.sku_id.map(gm)).sum())
    print(f'{c:>7.2f}{r.totals["working_capital_cost_eur"]:>20,.0f}{wc:>16,.0f}'
          f'{r.totals["lost_sales_eur"]:>16,.0f}{ls:>14,.0f}')

print('\nEvery cost reproduces from quantities x unit rates alone.')
print('No cost term reads the lever value. Guardrail holds.')

## The other three levers

Not breadth — the same one line, the same 15 SKUs. This is the gate criterion "four outputs responding to one lever", checked four times rather than once.

In [ ]:
sweeps = {}
for name, vals in [('service_target',           [0.90, 0.94, 0.96, 0.98, 0.995]),
                   ('inventory_cover_weeks',    [1.0, 2.0, 4.0, 7.0, 10.0]),
                   ('forecast_bias_correction', [0.0, 0.25, 0.5, 0.75, 1.0])]:
    sweeps[name] = engine.sweep(MVD_LINE, name, vals, base)
    print(f'\n=== {name}')
    print(sweeps[name][show].round(2).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.4))
for ax, (name, sw) in zip(axes, sweeps.items()):
    for col, lab in [('lost_sales_eur', 'lost sales'),
                     ('excess_obsolescence_eur', 'excess'),
                     ('working_capital_cost_eur', 'working capital'),
                     ('total_economic_cost_eur', 'TOTAL')]:
        ax.plot(sw.lever_value, sw[col] / 1e3, marker='o', lw=1.6,
                ls='-' if col == 'total_economic_cost_eur' else '--', label=lab)
    ax.set_title(name, fontsize=10); ax.set_xlabel(name); ax.grid(alpha=0.25)
axes[0].set_ylabel('EUR 000 / 12 months'); axes[-1].legend(fontsize=8)
plt.tight_layout(); plt.show()

## Confirming the write-off mechanism is live code, not dead code

The shelf-life write-off is structurally zero on L3, so this line alone cannot show that half of the output actually works. One run on L2 (chilled, 6-week shelf life) settles it.

This is **not** building breadth — no L2 result is claimed, no second line is added to scope. It is a liveness check on a code path the gate line cannot exercise.

In [ ]:
check = engine.run_scenario('L2', LeverSettings.defaults(assumptions, 'chilled', 'A'))
wo = float((check.sku_month.writeoff_units * check.sku_month.sku_id.map(std_cost)).sum())
print(f'L2 (chilled, 6-week shelf life)')
print(f'  shelf-life write-off   {wo:>14,.0f} EUR   <- mechanism fires')
print(f'  slow-moving provision  {check.totals["excess_obsolescence_eur"] - wo:>14,.0f} EUR')

## MVD gate — verdict

Fill this in against your own run, not against anything asserted in a chat.

| Criterion | Pass condition |
|---|---|
| Four outputs computed | all four finite and non-zero at the default lever setting |
| Four outputs respond | each of the four levers moves at least one output materially |
| Cross-lever coupling | `min_run_hours` moves conversion cost, carrying cost and excess in opposite directions |
| Trade-off is real | total economic cost has an interior optimum on at least one lever — neither slider end is best |
| Circularity guardrail | `tests/test_engine.py` passes, all tests |
| Capacity respected | workload never exceeds scheduled hours + max overtime |
| Reusable | the same code runs on `data_control` with no edits |

**Magnitudes are not findings (§10).** Every number above is whatever the generator and the assumption set encoded. What the gate demonstrates is that the mechanism runs end to end and the trade-off is real — not that lost sales on a personal care line are worth any particular figure.

In [ ]:
# Reusability: the same engine code, a different dataset, no edits.
# This needs data_control's own Step 5a output — the estimator must be re-run
# on the control panel, it cannot borrow the primary one. If you have not done
# that, this cell says so rather than reporting a green tick that means nothing.
if not os.path.isdir('data_control/raw'):
    sh('cp config/assumptions.yaml config/_bk.yaml')
    sh('cp config/assumptions_lowcensoring.yaml config/assumptions.yaml')
    sh('python generate_data.py')
    sh('cp config/_bk.yaml config/assumptions.yaml && rm config/_bk.yaml')
    sh('mv data data_control')

ing_c = DataIngestor(repo_root=REPO, data_root='data_control')
cm_c, sm_c, _ = DataCleaner(ing_c.schema, ing_c.assumptions).clean(ing_c.load())
os.makedirs('data_control/clean', exist_ok=True)
cm_c.to_parquet('data_control/clean/clean_master.parquet', index=False)
sm_c.to_parquet('data_control/clean/sku_master.parquet', index=False)

blind_cols = ["sku_id", "month", "actual_units", "forecast_l1_units",
              "forecast_l2_units", "forecast_l3_units", "production_units",
              "stock_close_units", "stock_open_units", "sched_adherence",
              "yield_rate", "promo_flag"]
cm_c[blind_cols].sort_values(["sku_id", "month"]).to_csv(
    'data_control/clean/clean_master_blind.csv', index=False)

if os.path.exists('data_control/clean/demand_characteristics.csv'):
    sh('IBP_DATA_ROOT=data_control python -m pytest tests/test_engine.py -q --no-header')
else:
    print('data_control/clean/clean_master_blind.csv written.')
    print()
    print('The control run is NOT yet demonstrated. To close it: run Step 5a on')
    print('that blind CSV, put its demand_characteristics.csv and')
    print('censoring_diagnostics.csv in data_control/clean/, and re-run this cell.')
    print('Reusability is a claim about a run you have done, not about code you')
    print('believe is general.')

## Save outputs to Drive
`files.download()` does not complete in this environment (D-028) — Drive mount instead.

In [ ]:
res.sku_month.to_csv('step6_sku_month.csv', index=False)
res.line_month.to_csv('step6_line_month.csv', index=False)
sweep.to_csv('step6_sweep_min_run_hours.csv', index=False)
for n, sw in sweeps.items():
    sw.to_csv(f'step6_sweep_{n}.csv', index=False)

from google.colab import drive
drive.mount('/content/drive')
import shutil, glob
out_dir = '/content/drive/My Drive/ibp-tradeoff-outputs'
os.makedirs(out_dir, exist_ok=True)
for f in glob.glob('step6_*.csv'):
    shutil.copy(f, os.path.join(out_dir, f))
print('copied to', out_dir)
print(sorted(os.path.basename(f) for f in glob.glob('step6_*.csv')))

## Immediately next — not in this notebook

**O-10 review.** SKU-to-line allocation is static and capacity-blind: `build_portfolio()` splits SKUs evenly by count with no regard for line speed. The MVD confirms the symptom directly — L3 runs around 35% utilisation against an 0.85 target, so `min_run_hours` never reaches the overtime step on this line and the capacity leg of the conversion-cost mechanism is untested. That review was committed to happen immediately after this gate, when cross-line evidence first exists. It is now due.